# MalariAI — Phase 2: Faster R-CNN Baseline (Kaggle)

**Kaysarul Anas Apurba · Laurentian University**  
Pipeline A · ResNet-50 FPN · NIH BBBC041 · 80 epochs across 2 sessions

---

### Session guide

| | Session 1 | Session 2 |
|---|---|---|
| `RESUME_FROM` | `None` | path to Session 1 `latest.pth` |
| Runs | Epochs 1 → 40 | Epochs 41 → 80 |
| After | Download outputs, upload as new dataset | Run mAP eval cell at end |

**Before running:**
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Attach your BBBC041 dataset (training.json + test.json + images/)
- If resuming: also attach the previous session's output dataset


## 1 · Confirm Dataset Path


In [ ]:
from pathlib import Path

# ── UPDATE THIS to match your Kaggle dataset slug ─────────────
DATASET_ROOT = Path('/kaggle/input/bbbc041-malaria')
# ──────────────────────────────────────────────────────────────

# Auto-detect if path is wrong
if not DATASET_ROOT.exists():
    kaggle_input = Path('/kaggle/input')
    dirs = sorted([d for d in kaggle_input.iterdir() if d.is_dir()])
    print('Dataset not found. Available inputs:')
    for d in dirs:
        print(f'  {d}')
    print('\nUpdate DATASET_ROOT above to the correct path, then re-run.')
else:
    print(f'Dataset found: {DATASET_ROOT}')
    for p in sorted(DATASET_ROOT.iterdir()):
        if p.is_dir():
            n = len(list(p.iterdir()))
            print(f'  {p.name}/  ({n} files)')
        else:
            print(f'  {p.name}')


## 2 · GPU & Dependencies


In [ ]:
import subprocess, sys

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchmetrics', '-q'], check=True)
    from torchmetrics.detection.mean_ap import MeanAveragePrecision

import torch
print(f'PyTorch  : {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU      : {props.name}  ({props.total_memory // 1024**3} GB VRAM)')
else:
    print('⚠  No GPU detected — enable it in Settings → Accelerator')
print('torchmetrics OK')


In [ ]:
import json, time, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms as T
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

print('All imports OK')


## 3 · Label Map


In [ ]:
LABEL_TO_INT = {
    'background':     0,
    'red blood cell': 1,
    'trophozoite':    2,
    'ring':           3,
    'schizont':       4,
    'gametocyte':     5,
    'leukocyte':      6,
}
INT_TO_LABEL     = {v: k for k, v in LABEL_TO_INT.items()}
NUM_CLASSES      = 7
PARASITE_CLASSES = ['trophozoite', 'ring', 'schizont', 'gametocyte']
SKIP_LABELS      = {'difficult'}

SHORT_NAME = {
    'red blood cell': 'RBC',   'trophozoite': 'Troph',
    'ring':           'Ring',  'schizont':    'Schiz',
    'gametocyte':     'Gam',   'leukocyte':   'Leuk',
    'background':     'BG',
}
CLASS_COLOUR_RGB = {
    'red blood cell': (220, 50,  50),  'trophozoite': (50,  180, 50),
    'ring':           (50,  50,  220), 'schizont':    (200, 130, 0),
    'gametocyte':     (160, 0,   200), 'leukocyte':   (0,   180, 200),
    'background':     (128, 128, 128),
}

print('Label map ready:')
for i in range(NUM_CLASSES):
    print(f'  {i}  {INT_TO_LABEL[i]}')


## 4 · Configuration

**This is the only cell you need to edit between sessions.**

- **Session 1:** `RESUME_FROM = None`  
- **Session 2:** set `RESUME_FROM` to the path of Session 1's `latest.pth`


In [ ]:
# ═══════════════════════════════════════════════════════════════
# EDIT THESE BETWEEN SESSIONS
# ═══════════════════════════════════════════════════════════════

RESUME_FROM    = None
# Session 2: RESUME_FROM = '/kaggle/input/phase2-session1/checkpoints/latest.pth'

TOTAL_EPOCHS   = 80    # total epochs across all sessions
SESSION_EPOCHS = 40    # how many to run this session

# ═══════════════════════════════════════════════════════════════

JSON_TRAIN  = DATASET_ROOT / 'training.json'
JSON_TEST   = DATASET_ROOT / 'test.json'
IMG_DIR     = DATASET_ROOT / 'images'

OUTPUT_DIR  = Path('/kaggle/working')
CKPT_DIR    = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CFG = dict(
    batch_size    = 4,       # T4 16 GB — fits 4 images at 1600×1200
    lr            = 5e-3,    # SGD initial LR (standard COCO recipe)
    momentum      = 0.9,
    weight_decay  = 1e-4,
    lr_step_epoch = 50,      # drop LR × 0.1 at global epoch 50
    lr_gamma      = 0.1,
    val_split     = 0.2,
    seed          = 42,
    num_workers   = 4,
    num_classes   = NUM_CLASSES,
    pretrained    = True,
    score_thresh  = 0.3,
)

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

resuming = RESUME_FROM is not None
print(f'Mode          : {"RESUME" if resuming else "FRESH START"}')
print(f'Session epochs: {SESSION_EPOCHS}')
print(f'Total target  : {TOTAL_EPOCHS} epochs')
print(f'LR drop at    : epoch {CFG["lr_step_epoch"]}')
print(f'Device        : {device}')


## 5 · Dataset


In [ ]:
class MalariaDataset(Dataset):
    """
    BBBC041 detection dataset — parses raw JSON directly.
    Returns (FloatTensor[3,H,W], {boxes, labels, image_id}).
    """
    def __init__(self, json_path, image_dir):
        self.image_dir = Path(image_dir)
        self.transform = T.ToTensor()
        self._records  = self._parse(Path(json_path))

    def _parse(self, json_path):
        with open(json_path) as f:
            raw = json.load(f)
        records = []
        for item in raw:
            img_name = Path(item['image']['pathname']).name
            boxes, labels = [], []
            for obj in item.get('objects', []):
                lbl = obj['category']
                if lbl in SKIP_LABELS or lbl not in LABEL_TO_INT:
                    continue
                bb = obj['bounding_box']
                x1 = float(bb['minimum']['c'])
                y1 = float(bb['minimum']['r'])
                x2 = float(bb['maximum']['c'])
                y2 = float(bb['maximum']['r'])
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(LABEL_TO_INT[lbl])
            if boxes:
                records.append({'img_name': img_name,
                                'boxes': boxes, 'labels': labels})
        return records

    def __len__(self): return len(self._records)

    def __getitem__(self, idx):
        rec    = self._records[idx]
        image  = Image.open(self.image_dir / rec['img_name']).convert('RGB')
        image  = self.transform(image)
        boxes  = torch.as_tensor(rec['boxes'],  dtype=torch.float32)
        labels = torch.as_tensor(rec['labels'], dtype=torch.int64)
        return image, {'boxes': boxes, 'labels': labels,
                       'image_id': torch.tensor([idx], dtype=torch.int64)}


def detection_collate(batch):
    return tuple(zip(*batch))


# ── Load & split ──────────────────────────────────────────────────────────
full_ds = MalariaDataset(JSON_TRAIN, IMG_DIR)
n_val   = int(len(full_ds) * CFG['val_split'])
n_train = len(full_ds) - n_val
gen     = torch.Generator().manual_seed(CFG['seed'])
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=gen)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], collate_fn=detection_collate,
                          pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], collate_fn=detection_collate,
                          pin_memory=True)

print(f'Train images : {len(train_ds):,}')
print(f'Val   images : {len(val_ds):,}')
print(f'Train batches: {len(train_loader):,}')

# Class distribution in training split
counts = defaultdict(int)
for idx in train_ds.indices:
    for lbl in full_ds._records[idx]['labels']:
        counts[INT_TO_LABEL[lbl]] += 1
total = sum(counts.values())
print(f'\nClass distribution ({total:,} total boxes in train):')
for cls in [INT_TO_LABEL[i] for i in range(1, NUM_CLASSES)]:
    cnt = counts.get(cls, 0)
    print(f'  {cls:<22} {cnt:>7,}  ({100*cnt/max(total,1):5.1f}%)')


## 6 · Visualise Training Samples


In [ ]:
def draw_boxes(ax, img_t, target, max_boxes=80, title=''):
    ax.imshow(img_t.permute(1, 2, 0).numpy())
    for i, (box, lbl) in enumerate(zip(
            target['boxes'].numpy(), target['labels'].numpy())):
        if i >= max_boxes: break
        cls = INT_TO_LABEL[lbl]
        r, g, b = CLASS_COLOUR_RGB[cls]
        ax.add_patch(patches.Rectangle(
            (box[0], box[1]), box[2]-box[0], box[3]-box[1],
            lw=1.2, edgecolor=(r/255,g/255,b/255), facecolor='none'))
        ax.text(box[0], max(box[1]-3,0), SHORT_NAME[cls], fontsize=5,
                color=(r/255,g/255,b/255),
                bbox=dict(facecolor='white', alpha=0.4, pad=0.5, linewidth=0))
    ax.set_title(f'{title}  ({len(target["boxes"])} boxes)', fontsize=8)
    ax.axis('off')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, idx in zip(axes.flat, random.sample(range(len(full_ds)), 6)):
    img, tgt = full_ds[idx]
    draw_boxes(ax, img, tgt, title=full_ds._records[idx]['img_name'][:20])
plt.suptitle('Training samples — ground-truth boxes (colour = class)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'sample_gt_boxes.png'), dpi=100, bbox_inches='tight')
plt.show()


## 7 · Model — Faster R-CNN ResNet-50 FPN

Pretrained on COCO. We replace only the 91-class head with a 7-class head.


In [ ]:
def build_model(num_classes, pretrained=True):
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if pretrained else None
    model   = fasterrcnn_resnet50_fpn(weights=weights)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features  # 1024
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    return model

model = build_model(CFG['num_classes'], pretrained=CFG['pretrained'])
model.to(device)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters : {total:,}  ({trainable:,} trainable)')
print(f'Head output: {model.roi_heads.box_predictor.cls_score.out_features} classes')
print(f'Device     : {next(model.parameters()).device}')


## 8 · Training Functions


In [ ]:
def train_one_epoch(model, loader, optimizer, device, global_epoch):
    model.train()
    total_loss = 0.0
    breakdown  = defaultdict(float)
    t0 = time.time()
    for i, (images, targets) in enumerate(loader):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses    = sum(loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        total_loss += losses.item()
        for k, v in loss_dict.items():
            breakdown[k] += v.item()
        if (i + 1) % 40 == 0:
            print(f'  [batch {i+1}/{len(loader)}]  '
                  f'loss={losses.item():.4f}  ({time.time()-t0:.0f}s)')
    n    = len(loader)
    comp = '  '.join(f'{k.replace("loss_","")}={v/n:.3f}'
                     for k, v in breakdown.items())
    return total_loss / n, comp


@torch.no_grad()
def validate_loss(model, loader, device):
    model.train()  # loss dict only available in train mode
    total = 0.0
    for images, targets in loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        total  += sum(model(images, targets).values()).item()
    return total / len(loader)


@torch.no_grad()
def compute_map(model, loader, device, score_thresh=0.3):
    model.eval()
    metric = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)
    for images, targets in loader:
        images = [img.to(device) for img in images]
        preds  = model(images)
        filtered = []
        for p in preds:
            keep = p['scores'] >= score_thresh
            filtered.append({'boxes':  p['boxes'][keep].cpu(),
                             'scores': p['scores'][keep].cpu(),
                             'labels': p['labels'][keep].cpu()})
        cpu_tgts = [{'boxes': t['boxes'].cpu(), 'labels': t['labels'].cpu()}
                    for t in targets]
        metric.update(filtered, cpu_tgts)
    return metric.compute()


print('Training functions defined.')


## 9 · Optimizer, Scheduler & Resume

**StepLR** drops the learning rate by 10× at global epoch 50, regardless of which session you are in.
The scheduler is fast-forwarded to the correct position when resuming.


In [ ]:
optimizer = optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG['lr'], momentum=CFG['momentum'], weight_decay=CFG['weight_decay'],
)
# MultiStepLR so the drop is tied to the GLOBAL epoch number, not session epoch
scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[CFG['lr_step_epoch']],  # drop at epoch 50
    gamma=CFG['lr_gamma'],              # × 0.1
)

# ── Resume from checkpoint ────────────────────────────────────────────────
start_epoch   = 1
train_losses  = []
val_losses    = []
best_val_loss = float('inf')

if RESUME_FROM is not None:
    print(f'Loading checkpoint: {RESUME_FROM}')
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch   = ckpt['epoch'] + 1
    train_losses  = ckpt.get('train_losses',  [])
    val_losses    = ckpt.get('val_losses',    [])
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    # Fast-forward scheduler to match global epoch
    for _ in range(ckpt['epoch']):
        scheduler.step()
    print(f'Resumed at epoch {start_epoch}')
    print(f'Best val_loss so far: {best_val_loss:.4f}')
    print(f'Current LR: {scheduler.get_last_lr()[0]:.2e}')
else:
    print('Fresh start — pretrained COCO weights loaded')
    print(f'Initial LR: {CFG["lr"]:.2e}')

end_epoch = start_epoch + SESSION_EPOCHS - 1
print(f'\nThis session: epochs {start_epoch} → {end_epoch}  (of {TOTAL_EPOCHS} total)')


## 10 · Training Loop

> ⏱ ~10–15 min/epoch on T4 → 40 epochs ≈ 7–10 hours.  
> The session lasts 12 hours. `latest.pth` is saved after every epoch.


In [ ]:
print(f'Starting training — epochs {start_epoch} to {end_epoch}')
print(f'{"Epoch":<8} {"Train Loss":<13} {"Val Loss":<13} {"LR":<12} Time')
print('-' * 60)

for epoch in range(start_epoch, end_epoch + 1):
    t0 = time.time()

    train_loss, comp = train_one_epoch(
        model, train_loader, optimizer, device, epoch)
    val_loss = validate_loss(model, val_loader, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    lr_now  = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    print(f'{epoch:<8} {train_loss:<13.4f} {val_loss:<13.4f} '
          f'{lr_now:<12.2e} {elapsed:.0f}s')
    print(f'         [{comp}]')

    ckpt = {
        'epoch':          epoch,
        'model':          model.state_dict(),
        'optimizer':      optimizer.state_dict(),
        'scheduler':      scheduler.state_dict(),
        'val_loss':       val_loss,
        'best_val_loss':  best_val_loss,
        'train_losses':   train_losses,
        'val_losses':     val_losses,
        'cfg':            CFG,
    }
    torch.save(ckpt, CKPT_DIR / 'latest.pth')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt['best_val_loss'] = best_val_loss
        torch.save(ckpt, CKPT_DIR / 'best.pth')
        print(f'         ✓ New best val_loss: {best_val_loss:.4f}')

print(f'\n--- Session complete: epochs {start_epoch}–{end_epoch} ---')
print(f'Best val_loss this run: {best_val_loss:.4f}')
if end_epoch < TOTAL_EPOCHS:
    print(f'\nNext session steps:')
    print(f'  1. Output tab → download checkpoints/ folder')
    print(f'  2. Upload as new Kaggle dataset (e.g. "phase2-session1")')
    print(f'  3. In the next session set:')
    print(f'     RESUME_FROM = "/kaggle/input/phase2-session1/checkpoints/latest.pth"')
else:
    print('All epochs complete — run the evaluation cells below.')


## 11 · Loss Curves


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
epochs_x = list(range(1, len(train_losses) + 1))
# If resuming, adjust x-axis to show global epochs
if RESUME_FROM:
    prev_epochs = start_epoch - 1
    epochs_x    = list(range(1, len(train_losses) + 1))  # cumulative history

ax.plot(epochs_x, train_losses, 'b-o', markersize=3, linewidth=1.2, label='Train loss')
ax.plot(epochs_x, val_losses,   'r-o', markersize=3, linewidth=1.2, label='Val loss')
ax.axvline(x=CFG['lr_step_epoch'], color='gray', linestyle='--',
           linewidth=1, label=f'LR drop (epoch {CFG["lr_step_epoch"]})')
ax.set_xlabel('Global Epoch')
ax.set_ylabel('Total loss')
ax.set_title('Faster R-CNN — Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'loss_curves.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/loss_curves.png')


## 12 · Evaluation — mAP@0.5 + Per-Class AP

Run this after **Session 2** (all 80 epochs done).  
These numbers are the **baseline row in Table 3** of the paper.


In [ ]:
print('Loading best checkpoint ...')
best_ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=device)
model.load_state_dict(best_ckpt['model'])
print(f'  Epoch       : {best_ckpt["epoch"]}')
print(f'  Val loss    : {best_ckpt["val_loss"]:.4f}')

print('\nRunning mAP@0.5 on validation split ...')
result = compute_map(model, val_loader, device,
                     score_thresh=CFG['score_thresh'])

map50 = result['map_50'].item()
print(f'\n{"="*52}')
print(f'  mAP@0.5 (all classes) : {map50:.4f}  ({100*map50:.1f}%)')
print(f'{"="*52}')

metrics = {
    'map_50':           round(map50, 4),
    'checkpoint_epoch': best_ckpt['epoch'],
    'val_loss':         round(best_ckpt['val_loss'], 4),
    'total_epochs':     TOTAL_EPOCHS,
    'cfg':              CFG,
}

if 'map_per_class' in result and result['map_per_class'] is not None:
    per = result['map_per_class'].tolist()
    metrics['per_class_ap'] = {}
    print(f'\n  {"Class":<22} {"AP@0.5":>8}  Note')
    print('  ' + '-'*44)
    for i, ap in enumerate(per):
        cls  = INT_TO_LABEL.get(i + 1, f'class_{i+1}')
        flag = '  ← rare parasite' if cls in PARASITE_CLASSES and cls != 'trophozoite' else ''
        metrics['per_class_ap'][cls] = round(ap, 4)
        print(f'  {cls:<22} {ap:>8.4f}{flag}')

with open(CKPT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nSaved → /kaggle/working/checkpoints/metrics.json')


## 13 · GT vs Predictions — Qualitative Check


In [ ]:
model.eval()
imgs_b, tgts_b = next(iter(val_loader))
imgs_gpu = [img.to(device) for img in imgs_b]
with torch.no_grad():
    preds = model(imgs_gpu)

n_show = min(4, len(imgs_b))
fig, axes = plt.subplots(2, n_show, figsize=(5*n_show, 7))
if n_show == 1: axes = axes.reshape(2, 1)

for col in range(n_show):
    img_np = imgs_b[col].permute(1, 2, 0).numpy()

    # Ground truth
    ax_gt = axes[0, col]
    ax_gt.imshow(img_np)
    for box, lbl in zip(tgts_b[col]['boxes'].numpy(),
                        tgts_b[col]['labels'].numpy()):
        cls = INT_TO_LABEL[lbl]
        r,g,b = CLASS_COLOUR_RGB[cls]
        ax_gt.add_patch(patches.Rectangle(
            (box[0],box[1]), box[2]-box[0], box[3]-box[1],
            lw=1.2, edgecolor=(r/255,g/255,b/255), facecolor='none'))
    ax_gt.set_title(f'GT  ({len(tgts_b[col]["boxes"])} boxes)', fontsize=8)
    ax_gt.axis('off')

    # Predictions
    ax_p = axes[1, col]
    ax_p.imshow(img_np)
    p    = preds[col]
    keep = p['scores'].cpu() >= CFG['score_thresh']
    for box, lbl, sc in zip(p['boxes'][keep].cpu().numpy(),
                            p['labels'][keep].cpu().numpy(),
                            p['scores'][keep].cpu().numpy()):
        cls = INT_TO_LABEL.get(lbl, 'background')
        r,g,b = CLASS_COLOUR_RGB.get(cls, (128,128,128))
        ax_p.add_patch(patches.Rectangle(
            (box[0],box[1]), box[2]-box[0], box[3]-box[1],
            lw=1.2, edgecolor=(r/255,g/255,b/255), facecolor='none'))
        ax_p.text(box[0], max(box[1]-3,0),
                  f'{SHORT_NAME.get(cls,"?")} {sc:.2f}',
                  fontsize=4.5, color=(r/255,g/255,b/255))
    ax_p.set_title(f'Pred ({keep.sum().item()} boxes, ≥{CFG["score_thresh"]})', fontsize=8)
    ax_p.axis('off')

plt.suptitle('Ground Truth (top) vs Faster R-CNN Predictions (bottom)', fontsize=10)
plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'gt_vs_pred.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Saved → checkpoints/gt_vs_pred.png')


## 14 · Output Summary — What to Download

All files are in the **Output** tab under `checkpoints/`.

| File | What it is |
|---|---|
| `best.pth` | Best model by val loss — keep this permanently |
| `latest.pth` | Last epoch checkpoint — use for Session 2 resume |
| `metrics.json` | **Baseline row for Table 3 of the paper** |
| `loss_curves.png` | Training figure for §4 Experiments |
| `gt_vs_pred.png` | Qualitative figure for §5 Results |
| `sample_gt_boxes.png` | Dataset sanity check |

**After both sessions are done → copy into your local repo:**
```
Phase2-BaselineA/checkpoints/best.pth
Phase2-BaselineA/checkpoints/metrics.json
```


In [ ]:
import os
print('Files in /kaggle/working/checkpoints/')
print('-' * 45)
for f in sorted(CKPT_DIR.iterdir()):
    size = os.path.getsize(f)
    if size > 1024*1024:
        print(f'  {f.name:<35} {size/1024/1024:6.1f} MB')
    else:
        print(f'  {f.name:<35} {size/1024:6.1f} KB')
